In [ ]:
from io import StringIO

import pandas as pd
import requests
import yfinance as yf


## Exercise 1

In [ ]:
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/91.0.4472.124 Safari/537.36'
    )
}

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

# read_html parses every HTML table found in the downloaded page.
tables = pd.read_html(StringIO(response.text))
sp500 = tables[0]

print(f'Found {len(tables)} tables')
print(sp500.shape)
sp500.head()

Found 2 tables
(503, 8)


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


**Which year had the highest number of additions (starting from 2020)?** 

so which year since 2020 - 6 options


In [6]:
# 1. Convert 'Date added' to datetime
sp500['Date added'] = pd.to_datetime(sp500['Date added'])

# 2. Filter for years strictly greater than 2019 and count the occurrences
recent_additions = sp500[sp500['Date added'].dt.year > 2019]['Date added'].dt.year.value_counts()

# 3. Get the top year and its count
most_common_year = recent_additions.index[0]
count = recent_additions.iloc[0]

print(f"Starting in 2020, the year that appears the most is {most_common_year} with {count} additions.")


Starting in 2020, the year that appears the most is 2025 with 18 additions.


### Question 2. [Macro] Indexes YTD (as of 21 August 2026)

**How many indexes (out of 10) have better year-to-date returns than the US (S&P 500) as of August 21, 2026?**

Using Yahoo Finance World Indices data, compare the year-to-date (YTD) performance (1 January-21 August 2026, use the closing price to calculate the growth) of major stock market indexes for the following countries:
* United States - S&P 500 (^GSPC)
* China - Shanghai Composite (000001.SS)
* Hong Kong - HANG SENG INDEX (^HSI)	
* Australia - S&P/ASX 200 (^AXJO)
* India - Nifty 50 (^NSEI)
* Canada - S&P/TSX Composite (^GSPTSE)
* Germany - DAX (^GDAXI)
* United Kingdom - FTSE 100 (^FTSE)
* Japan - Nikkei 225 (^N225)
* Mexico - IPC Mexico (^MXX)
* Brazil - Ibovespa (^BVSP)

In [9]:
# 1. Define the dictionary of indexes and their tickers
index_dict = {
    "United States (^GSPC)": "^GSPC",
    "China (000001.SS)": "000001.SS",
    "Hong Kong (^HSI)": "^HSI",
    "Australia (^AXJO)": "^AXJO",
    "India (^NSEI)": "^NSEI",
    "Canada (^GSPTSE)": "^GSPTSE",
    "Germany (^GDAXI)": "^GDAXI",
    "United Kingdom (^FTSE)": "^FTSE",
    "Japan (^N225)": "^N225",
    "Mexico (^MXX)": "^MXX",
    "Brazil (^BVSP)": "^BVSP",
}

results = []

print("Fetching data and calculating individual returns...")
for name, ticker in index_dict.items():
    df = yf.download(ticker, start="2026-01-01", end="2026-08-22", progress=False)
    
    if not df.empty and "Close" in df.columns:
        # Extract just the raw closing prices as a flat array of numbers
        close_prices = df["Close"].values.flatten()
        
        # Drop any NaN values that might have sneaked into the price array
        close_prices = close_prices[~pd.isna(close_prices)]
        
        if len(close_prices) >= 2:
            first_price = float(close_prices[0])
            last_price = float(close_prices[-1])
            
            # Calculate percent growth
            ytd_return = ((last_price - first_price) / first_price) * 100
            results.append({"Index Name": name, "YTD Return (%)": ytd_return})
        else:
            results.append({"Index Name": name, "YTD Return (%)": None})
    else:
        results.append({"Index Name": name, "YTD Return (%)": None})

# 2. Create DataFrame and sort
results_df = pd.DataFrame(results).set_index("Index Name")
results_df = results_df.dropna().sort_values(by="YTD Return (%)", ascending=False)

# 3. Extract S&P 500 performance to calculate the final answer dynamically
sp500_return = results_df.loc["United States (^GSPC)", "YTD Return (%)"]
better_than_us = results_df[results_df["YTD Return (%)"] > sp500_return]

# 4. Print final cleaner output
print("\n--- Index YTD Returns (sorted) ---")
print(results_df.round(2))
print("\n----------------------------------")
print(
    f"Number of indexes with better YTD returns than the US S&P 500: {len(better_than_us)} out of 10"
)

Fetching data and calculating individual returns...

--- Index YTD Returns (sorted) ---
                        YTD Return (%)
Index Name                            
Japan (^N225)                    27.36
Canada (^GSPTSE)                 14.86
United States (^GSPC)            11.90
United Kingdom (^FTSE)            8.70
Brazil (^BVSP)                    6.54
Germany (^GDAXI)                  6.51
Australia (^AXJO)                 3.79
Mexico (^MXX)                     2.48
Hong Kong (^HSI)                 -1.25
China (000001.SS)                -2.94
India (^NSEI)                    -7.25

----------------------------------
Number of indexes with better YTD returns than the US S&P 500: 2 out of 10


In [10]:
# 1. Define tickers matching your assignment
index_dict = {
    "United States (^GSPC)": "^GSPC",
    "China (000001.SS)": "000001.SS",
    "Hong Kong (^HSI)": "^HSI",
    "Australia (^AXJO)": "^AXJO",
    "India (^NSEI)": "^NSEI",
    "Canada (^GSPTSE)": "^GSPTSE",
    "Germany (^GDAXI)": "^GDAXI",
    "United Kingdom (^FTSE)": "^FTSE",
    "Japan (^N225)": "^N225",
    "Mexico (^MXX)": "^MXX",
    "Brazil (^BVSP)": "^BVSP",
}

# Target evaluation date
end_date = "2026-08-22"  # Safely covers August 21 trading close
horizons = {"3-Year": "2023-08-21", "5-Year": "2021-08-21", "10-Year": "2016-08-21"}

results_summary = []

print("Running historical multi-period analysis...")
for name, ticker in index_dict.items():
    # Fetch data back to 2016 to cover all horizons in one clean go
    df = yf.download(ticker, start="2016-08-15", end=end_date, progress=False)

    if not df.empty and "Close" in df.columns:
        # Get raw close data stripped of multi-index issues
        close_series = df["Close"].iloc[:, 0] if isinstance(df["Close"], pd.DataFrame) else df["Close"]
        close_series = close_series.dropna()
        
        # Pull final closing price (Aug 21, 2026)
        final_price = float(close_series.iloc[-1])
        row = {"Index": name}
        
        for label, start_date in horizons.items():
            # Get closest available trading day to target historical start date
            historical_data = close_series.loc[start_date:]
            if not historical_data.empty:
                start_price = float(historical_data.iloc[0])
                returns = ((final_price - start_price) / start_price) * 100
                row[label] = returns
            else:
                row[label] = None
        results_summary.append(row)

# 2. Structure into a scannable DataFrame
perf_df = pd.DataFrame(results_summary).set_index("Index")

# 3. Check performance trends against US baseline
us_key = "United States (^GSPC)"
for col in horizons.keys():
    us_return = perf_df.loc[us_key, col]
    # Count how many other rows have a higher return value
    better_count = (perf_df[col] > us_return).sum()
    print(f"\n🏆 Over the {col} window: {better_count} out of 10 international indexes beat the S&P 500.")

print("\n--- Detailed Historical Performance Return (%) ---")
print(perf_df.round(2))

Running historical multi-period analysis...

🏆 Over the 3-Year window: 2 out of 10 international indexes beat the S&P 500.

🏆 Over the 5-Year window: 2 out of 10 international indexes beat the S&P 500.

🏆 Over the 10-Year window: 1 out of 10 international indexes beat the S&P 500.

--- Detailed Historical Performance Return (%) ---
                        3-Year  5-Year  10-Year
Index                                          
United States (^GSPC)    74.43   71.32   251.61
China (000001.SS)        26.26   12.31    26.59
Hong Kong (^HSI)         47.59    3.58    13.09
Australia (^AXJO)        27.31   20.95    64.26
India (^NSEI)            25.05   47.01   181.05
Canada (^GSPTSE)         85.09   78.83   148.30
Germany (^GDAXI)         67.51   64.87   149.05
United Kingdom (^FTSE)   49.03   52.15    58.40
Japan (^N225)           109.14  140.11   297.73
Mexico (^MXX)            23.76   26.27    36.10
Brazil (^BVSP)           49.47   45.59   196.00


### Question 3. [Index] S&P 500 Market Corrections Analysis


**Calculate the median drawdown (in %) of significant market corrections in the S&P 500 index.**

For this task, define a correction as an event when a stock index goes down by **at least 5%** from the most recent all-time high.

Steps:
1. Download S&P 500 historical data (period from 1950 to present) using yfinance (daily stats)
2. Identify all-time high closing price points (where price exceeds all previous days' prices)
3. For each pair of consecutive all-time highs, find the minimum price in between
4. Calculate drawdown percentages: (high - low) / high × 100
5. Filter for corrections with at least 5% drawdown
6. Calculate the duration in days for each correction period
7. Determine the 25th, 50th (median), and 75th percentiles for correction durations and drawdowns

In [14]:
# Step 1: Download S&P 500 historical data (1950 to present)
print("Downloading historical S&P 500 daily statistics...")
df = yf.download("^GSPC", start="1950-01-01", progress=False)

# Clean multi-index columns if present in newer yfinance versions
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Work with clean closing prices
prices = df["Close"].dropna()

# Step 2: Identify all-time high closing price points and their index dates
ath_series = prices.cummax()
ath_dates = ath_series[prices == ath_series].index

drawdowns = []
durations = []

# Step 3 & 4: Evaluate the minimum price between pairs of consecutive all-time highs
for i in range(len(ath_dates) - 1):
    high_date = ath_dates[i]
    next_high_date = ath_dates[i + 1]

    high_price = float(prices.loc[high_date])

    # Slice the sub-period from this ATH to the next ATH
    sub_period = prices.loc[high_date:next_high_date]

    if len(sub_period) > 1:
        min_price = float(sub_period.min())

        # Calculate drawdown percentage
        dd = ((high_price - min_price) / high_price) * 100

        # Step 5 & 6: Filter for corrections >= 5% and record high-to-low duration
        if dd >= 5.0:
            drawdowns.append(dd)
            # Find the date where the low occurred to calculate the exact duration
            min_date = sub_period.idxmin()
            duration_days = (min_date - high_date).days
            durations.append(duration_days)

# Create a summary DataFrame for percentiles
results_df = pd.DataFrame(
    {"Drawdown (%)": drawdowns, "Duration (Days)": durations}
)

# Step 7: Determine the 25th, 50th, and 75th percentiles
percentiles = [0.25, 0.50, 0.75, 0.95, 0.99]
summary = results_df.quantile(percentiles)
summary.index = ["25th Percentile", "50th Percentile (Median)", "75th Percentile", "95th Percentile", "99th Percentile"]

print("\n--- S&P 500 Market Corrections Analysis Results ---")
print(summary.round(2))


--- S&P 500 Market Corrections Analysis Results ---
                          Drawdown (%)  Duration (Days)
25th Percentile                   6.23            22.00
50th Percentile (Median)          7.99            40.50
75th Percentile                  14.02            86.25
95th Percentile                  34.67           526.10
99th Percentile                  51.21           710.73


In [15]:
# 1. Download S&P 500 historical data
print("Downloading historical S&P 500 daily statistics...")
df = yf.download("^GSPC", start="1950-01-01", progress=False)

# Clean multi-index columns if present in newer yfinance versions
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

prices = df["Close"].dropna()

# 2. Identify all-time high points
ath_series = prices.cummax()
ath_dates = ath_series[prices == ath_series].index

correction_records = []

# 3 & 4: Evaluate minimum price between consecutive ATHs
for i in range(len(ath_dates) - 1):
    high_date = ath_dates[i]
    next_high_date = ath_dates[i + 1]

    high_price = float(prices.loc[high_date])
    sub_period = prices.loc[high_date:next_high_date]

    if len(sub_period) > 1:
        min_price = float(sub_period.min())
        dd = ((high_price - min_price) / high_price) * 100

        # 5 & 6: Filter for corrections >= 5% and record dates/durations
        if dd >= 5.0:
            min_date = sub_period.idxmin()
            duration_days = (min_date - high_date).days
            
            correction_records.append({
                "ATH Date": high_date.strftime('%Y-%m-%d'),
                "Bottom Date": min_date.strftime('%Y-%m-%d'),
                "Drawdown (%)": dd,
                "Duration (Days)": duration_days
            })

# Convert to DataFrame
results_df = pd.DataFrame(correction_records)

# Calculate expanded target percentiles for drawdowns
target_percentiles = [0.25, 0.50, 0.75, 0.95, 0.99]
percentile_values = {p: results_df["Drawdown (%)"].quantile(p) for p in target_percentiles}

# Function to find the most recent correction closest to a target drawdown value
def get_closest_recent_correction(target_val):
    df_copy = results_df.copy()
    df_copy["distance"] = (df_copy["Drawdown (%)"] - target_val).abs()
    
    # Sort by closest distance, then by date to find the most recent one
    df_copy = df_copy.sort_values(by=["distance", "Bottom Date"], ascending=[True, False])
    return df_copy.iloc[0]

# Compile summary entries
summary_rows = []
labels = {
    0.25: "25th Percentile (Common Routine)",
    0.50: "50th Percentile (Standard Median)",
    0.75: "75th Percentile (Serious Correction)",
    0.95: "95th Percentile (Severe Crash/Bear Market)",
    0.99: "99th Percentile (Historic Collapse)"
}

for p in target_percentiles:
    p_val = percentile_values[p]
    closest_match = get_closest_recent_correction(p_val)
    summary_rows.append({
        "Percentile Group": labels[p],
        "Theoretical Threshold (%)": round(p_val, 2),
        "Most Recent Match Date": closest_match["Bottom Date"],
        "Actual Drawdown (%)": round(closest_match["Drawdown (%)"], 2),
        "Days to Trough": int(closest_match["Duration (Days)"])
    })

# Print Summary Table
print("\n--- S&P 500 Expanded Market Corrections Analysis Results ---")
print(pd.DataFrame(summary_rows).to_string(index=False))


--- S&P 500 Expanded Market Corrections Analysis Results ---
                          Percentile Group  Theoretical Threshold (%) Most Recent Match Date  Actual Drawdown (%)  Days to Trough
          25th Percentile (Common Routine)                       6.23             1961-04-24                 6.23               7
         50th Percentile (Standard Median)                       7.99             1951-06-29                 8.11              57
      75th Percentile (Serious Correction)                      14.02             1950-07-17                14.02              35
95th Percentile (Severe Crash/Bear Market)                      34.67             2020-03-23                33.92              33
       99th Percentile (Historic Collapse)                      51.21             2002-10-09                49.15             929


### Question 4.  [Stocks] Earnings Surprise Analysis for Amazon (AMZN)


**Calculate the median 2-day percentage change in stock prices following positive earnings surprise days.**


In [17]:
# Step 1: Load earnings data for AMZN
print("Fetching Amazon earnings history...")
ticker = "AMZN"
ticker_obj = yf.Ticker(ticker)
earnings_df = ticker_obj.get_earnings_dates()

# Clean MultiIndex columns if present in get_earnings_dates()
if isinstance(earnings_df.columns, pd.MultiIndex):
    earnings_df.columns = earnings_df.columns.get_level_values(-1)

# Standardize column names to remove extra spaces or case mismatches
earnings_df.columns = earnings_df.columns.str.strip()

# Find the exact surprise column name dynamically (handles 'Surprise %' or 'Surprise(%)')
surprise_col = [col for col in earnings_df.columns if "Surprise" in col][0]
reported_col = [col for col in earnings_df.columns if "Reported" in col][0]

# Filter out future dates that don't have surprise metrics filled in
earnings_df = earnings_df.dropna(subset=[surprise_col])

# Clean up index timezones to allow perfect matching with daily price dates
earnings_df.index = earnings_df.index.tz_localize(None).normalize()

# Step 2: Download complete historical price data
print("Downloading historical price data...")
price_df = yf.download(ticker, start="2020-01-01", progress=False)

# Clean multi-index columns for historical prices if present
if isinstance(price_df.columns, pd.MultiIndex):
    price_df.columns = price_df.columns.get_level_values(0)

# Step 3: Calculate 2-day percentage changes for all dates
# If current row is Day 2 (Earnings Day), shift(1) is Day 1, and shift(-1) is Day 3
price_df["2Day_Return"] = (price_df["Close"].shift(-1) / price_df["Close"].shift(1)) - 1

# Step 4: Merge earnings surprise metrics with the stock returns
merged_df = earnings_df.join(price_df[["2Day_Return"]], how="inner")

# Convert the surprise column to a numeric type safely
merged_df[surprise_col] = pd.to_numeric(merged_df[surprise_col], errors="coerce")

# Filter for strictly positive earnings surprises
positive_surprises = merged_df[merged_df[surprise_col] > 0]

# Calculate metrics
median_return = positive_surprises["2Day_Return"].median() * 100
correlation = positive_surprises["2Day_Return"].corr(positive_surprises[surprise_col])

# Print results
print("\n--- Amazon (AMZN) Earnings Surprise Analysis ---")
print(f"Total positive surprise events analyzed: {len(positive_surprises)}")
print(f"Median 2-day stock return after positive surprise: {median_return:.2f}%")
print(f"Correlation between surprise magnitude and 2-day return: {correlation:.4f}")

# View the core underlying data sorted by highest surprise magnitude
print("\n--- Sample Breakdown (Top 5 Positive Surprises) ---")
print(positive_surprises[[reported_col, surprise_col, "2Day_Return"]].sort_values(by=surprise_col, ascending=False).head(5))


Fetching Amazon earnings history...

--- Amazon (AMZN) Earnings Surprise Analysis ---
Total positive surprise events analyzed: 20
Median 2-day stock return after positive surprise: 0.35%
Correlation between surprise magnitude and 2-day return: 0.3306

--- Sample Breakdown (Top 5 Positive Surprises) ---
               Reported EPS  Surprise(%)  2Day_Return
Earnings Date                                        
2022-02-03             1.39       657.12     0.046656
2026-07-30             5.75       215.02     0.198235
2021-02-02             0.70       100.10    -0.009079
2023-08-03             0.65        91.19     0.088605
2026-04-29             2.78        69.02     0.020639


In [18]:
# Step 1: Load earnings data for AMZN
print("Fetching Amazon earnings history...")
ticker = "AMZN"
ticker_obj = yf.Ticker(ticker)
earnings_df = ticker_obj.get_earnings_dates()

# Clean MultiIndex columns if present in get_earnings_dates()
if isinstance(earnings_df.columns, pd.MultiIndex):
    earnings_df.columns = earnings_df.columns.get_level_values(-1)

# Standardize column names to remove extra spaces or case mismatches
earnings_df.columns = earnings_df.columns.str.strip()

# Find the exact surprise column name dynamically (handles 'Surprise %' or 'Surprise(%)')
surprise_col = [col for col in earnings_df.columns if "Surprise" in col][0]
reported_col = [col for col in earnings_df.columns if "Reported" in col][0]

# Filter out future dates that don't have surprise metrics filled in
earnings_df = earnings_df.dropna(subset=[surprise_col])

# Clean up index timezones to allow perfect matching with daily price dates
earnings_df.index = earnings_df.index.tz_localize(None).normalize()

# Step 2: Download complete historical price data
print("Downloading historical price data...")
price_df = yf.download(ticker, start="2020-01-01", progress=False)

# Clean multi-index columns for historical prices if present
if isinstance(price_df.columns, pd.MultiIndex):
    price_df.columns = price_df.columns.get_level_values(0)

# Step 3: Calculate 2-day percentage changes for all dates
# If current row is Day 2 (Earnings Day), shift(1) is Day 1, and shift(-1) is Day 3
price_df["2Day_Return"] = (price_df["Close"].shift(-1) / price_df["Close"].shift(1)) - 1

# Step 4: Merge earnings surprise metrics with the stock returns
merged_df = earnings_df.join(price_df[["2Day_Return"]], how="inner")

# Convert the surprise column to a numeric type safely
merged_df[surprise_col] = pd.to_numeric(merged_df[surprise_col], errors="coerce")

# Filter for strictly positive earnings surprises
positive_surprises = merged_df[merged_df[surprise_col] > 0]

# Calculate metrics
median_return = positive_surprises["2Day_Return"].median() * 100
correlation = positive_surprises["2Day_Return"].corr(positive_surprises[surprise_col])

# Print results
print("\n--- Amazon (AMZN) Earnings Surprise Analysis ---")
print(f"Total positive surprise events analyzed: {len(positive_surprises)}")
print(f"Median 2-day stock return after positive surprise: {median_return:.2f}%")
print(f"Correlation between surprise magnitude and 2-day return: {correlation:.4f}")

# View the core underlying data sorted by highest surprise magnitude
print("\n--- Sample Breakdown (Top 5 Positive Surprises) ---")
print(positive_surprises[[reported_col, surprise_col, "2Day_Return"]].sort_values(by=surprise_col, ascending=False).head(5))


# =====================================================================
# ADDITIONAL REGIME ANALYSIS: BULL VS. BEAR MARKETS
# =====================================================================

print("\n--- Market Regime Analysis (Bull vs. Bear) ---")

# 1. Split the positive surprise dataset into distinct market regimes
bear_2022 = positive_surprises[positive_surprises.index.year == 2022]
bull_recent = positive_surprises[positive_surprises.index.year.isin([2023, 2024, 2025, 2026])]

# 2. Process the 2022 Bear Market Regime
if len(bear_2022) >= 2:
    bear_corr = bear_2022["2Day_Return"].corr(bear_2022[surprise_col])
    bear_median = bear_2022["2Day_Return"].median() * 100
    print(f"🔴 2022 Bear Market (Count: {len(bear_2022)}):")
    print(f"   - Median Stock Return: {bear_median:.2f}%")
    print(f"   - Surprise Correlation: {bear_corr:.4f}")
else:
    print("🔴 2022 Bear Market: Insufficient data points to calculate correlation.")

print("-" * 46)

# 3. Process the Recent Bull Market Regime
if len(bull_recent) >= 2:
    bull_corr = bull_recent["2Day_Return"].corr(bull_recent[surprise_col])
    bull_median = bull_recent["2Day_Return"].median() * 100
    print(f"🟢 Recent Bull Market (Count: {len(bull_recent)}):")
    print(f"   - Median Stock Return: {bull_median:.2f}%")
    print(f"   - Surprise Correlation: {bull_corr:.4f}")
else:
    print("🟢 Recent Bull Market: Insufficient data points to calculate correlation.")


Fetching Amazon earnings history...

--- Amazon (AMZN) Earnings Surprise Analysis ---
Total positive surprise events analyzed: 20
Median 2-day stock return after positive surprise: 0.35%
Correlation between surprise magnitude and 2-day return: 0.3306

--- Sample Breakdown (Top 5 Positive Surprises) ---
               Reported EPS  Surprise(%)  2Day_Return
Earnings Date                                        
2022-02-03             1.39       657.12     0.046656
2026-07-30             5.75       215.02     0.198235
2021-02-02             0.70       100.10    -0.009079
2023-08-03             0.65        91.19     0.088605
2026-04-29             2.78        69.02     0.020639

--- Market Regime Analysis (Bull vs. Bear) ---
🔴 2022 Bear Market (Count: 2):
   - Median Stock Return: -2.96%
   - Surprise Correlation: 1.0000
----------------------------------------------
🟢 Recent Bull Market (Count: 14):
   - Median Stock Return: 2.38%
   - Surprise Correlation: 0.7384
